### 01. Environment Setup
> *Initialize libraries and project-wide constants for data paths and API identifiers.*

In [13]:
import os
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

# Dataset Identifier
DATASET = "patateriedata/all-international-football-results"

# Directory Structure
RAW_DIR = os.path.join("data", "raw")
PROCESSED_DIR = os.path.join("data", "processed")

# File Paths
MATCHES_RAW = os.path.join(RAW_DIR, "all_matches.csv")
COUNTRIES_REF = os.path.join(RAW_DIR, "countries_names.csv")
OUTPUT_CLEAN = os.path.join(PROCESSED_DIR, "cleaned_matches.csv")

### 02. Data Ingestion
> *Download the dataset from Kaggle into the raw directory. This step is skipped if the files already exist.*

In [14]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

if not os.path.exists(MATCHES_RAW):
    print("Fetching dataset from Kaggle...")
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(DATASET, path=RAW_DIR, unzip=True)
    print("Download complete.")
else:
    print("Raw dataset already exists.")

Fetching dataset from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/patateriedata/all-international-football-results
Download complete.


### 03. Cleaning and Standardization
> *Standardize team names by stripping whitespace and applying a current name mapping for historical consistency.*

In [15]:
# Load data from raw directory
matches = pd.read_csv(MATCHES_RAW, parse_dates=["date"])
countries = pd.read_csv(COUNTRIES_REF)

# Basic cleaning
for df in [matches, countries]:
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes('object'):
        df[col] = df[col].str.strip()

countries = countries.drop_duplicates(subset=['original_name'])

# Standardize team names
for side in ['home', 'away']:
    col = f'{side}_team'
    matches = matches.merge(
        countries[['original_name', 'current_name']],
        left_on=col,
        right_on='original_name',
        how='left'
    )
    # Replace with standardized name if found
    matches[col].update(matches['current_name'])
    matches.drop(columns=['original_name', 'current_name'], inplace=True)

print(f"Processing complete. Final dataset contains {len(matches)} matches.")
display(matches.head())

Processing complete. Final dataset contains 51270 matches.


,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


### 04. Export Results
> *Save the cleaned matches to the processed directory for analytical use.*

In [16]:
matches.to_csv(OUTPUT_CLEAN, index=False)
print(f"File saved successfully to: {OUTPUT_CLEAN}")

File saved successfully to: data\processed\cleaned_matches.csv
